# Training on the NVD/OSV Dataset

## using labeled.csv

---

Data Loading & Preparation

In [ ]:
import os
import kagglehub
import pandas as pd


In [ ]:
from nltk.stem import PorterStemmer

def clean(df: pd.DataFrame):
    
    # open up stemmer
    stemmer = PorterStemmer()

    # drop null text rows
    original_length = len(df)
    df = df.dropna(subset=["text"]).copy()
    print(f"Dropped null text rows: {original_length - len(df)} rows dropped.")

    # keep first occurrence of each record and drop duplicates
    before = len(df)
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    print(f"Dropped duplicate rows: {before - len(df)} rows dropped.\n")
    
    rows_remaining = len(df)
    rows_dropped = original_length - rows_remaining
    percent_remaining = (rows_remaining / original_length) * 100

    print(f"Done. {rows_dropped} total rows dropped.")
    print(f"{rows_remaining}/{original_length} {percent_remaining:.1f}% of rows remaining ")

    # fill null labels with -1
    df["target"] = df["target"].fillna(-1)

    # lowercase
    df["text"] = df["text"].str.lower()

    return df

    # remove URLs, domains, and file extensions
    #df["text"] = df["text"].str.replace(
    #    r"http\S+|www\.\S+|\S+\.(com|org|net|gov|html|php|php\d|txt|asc|xml|cgi)\S*|\d+",
    #    "", regex=True
    #)

    # nuclear option. remove anything that isn't a letter or space
    #df["text"] = df["text"].str.replace(r"[^a-z\s]", "", regex=True)
    # stem words + remove duplicate words
    stemmed_texts = []
    total = len(df)
    last_printed = -1

    for i, text in enumerate(df["text"]):
        if isinstance(text, str):
            
            stemmed_words = {}
            
            for w in text.split():
                stemmed_words[stemmer.stem(w)] = None
            
            stemmed_texts.append(" ".join(stemmed_words.keys()))
        
        else:
            stemmed_texts.append(text)

        percent = int((i + 1) / total * 100)
        if percent % 5 == 0 and percent != last_printed:
            print(f"   Processing. {i + 1}/{total} completed \t({percent}%)")
            last_printed = percent

    df["text"] = stemmed_texts
    
    return df

In [ ]:
import pandas as pd

def extract_primary_libraries(df: pd.DataFrame) -> pd.DataFrame:
    library_ids_list = []
    for x in df['target']:

        if pd.notna(x):
            parsed_ids = []

            for i in str(x).split():
                parsed_ids.append(int(i))
            library_ids_list.append(parsed_ids)

        else:
            library_ids_list.append([])

    df['library_ids'] = library_ids_list

    # Extract primary library (first in list)
    primary_library_list = []

    for x in df['library_ids']:
        if len(x) > 0:
            primary_library_list.append(x[0])
        else:
            primary_library_list.append(-1)

    df['primary_library'] = primary_library_list

    # Count affected libraries
    num_libraries_list = []
    for x in df['library_ids']:
        num_libraries_list.append(len(x))
    df['num_libraries'] = num_libraries_list

    # Map primary libraries to unique, contiguous labels
    unique_labels = sorted(df['primary_library'].unique())
    label_map = {label: i for i, label in enumerate(unique_labels)}
    df['label'] = df['primary_library'].map(label_map)

    return df

# Download latest version
path = kagglehub.dataset_download("dngchnhtrn/nvd-cve-reports-and-affected-libraries")
df = pd.read_csv(os.path.join(path, "labeled.csv"))
df = clean(df)
df = extract_primary_libraries(df)


In [ ]:
print(df.head)

In [ ]:
from sklearn.model_selection import train_test_split


counts = df['label'].value_counts()

# Split into single-sample and multi-sample labels
single_sample = df[df['label'].isin(counts[counts == 1].index)]
multi_sample  = df[df['label'].isin(counts[counts > 1].index)]

# Stratified split on multi-sample only
train_multi, test = train_test_split(
    multi_sample,
    test_size=0.2,
    stratify=multi_sample['label'],
    random_state=42
)

# Add single-sample labels back into train only
train = pd.concat([train_multi, single_sample])

X_train = train['text']
y_train = train['label']
X_test  = test['text']
y_test  = test['label']

print(f"Train samples: {len(X_train)}")
print(f"Test samples:  {len(X_test)}")
print(f"Train labels:  {y_train.nunique()}")
print(f"Test labels:   {y_test.nunique()}")

In [46]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, confusion_matrix, f1_score, multilabel_confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display
import numpy as np

pipeline_tfidf = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, sublinear_tf=True, ngram_range=(1, 1)))
])
X_train_tfidf = pipeline_tfidf.fit_transform(X_train)
X_test_tfidf  = pipeline_tfidf.transform(X_test)

k_neighbors = 10
knn = KNeighborsClassifier(n_neighbors=k_neighbors, metric='cosine')
knn.fit(X_train_tfidf, y_train)

distances, indices = knn.kneighbors(X_test_tfidf)
neighbor_labels = np.array(y_train)[indices]  # shape: (n_test_samples, k)

# For each test sample, check if true label is in top-k predictions
y_test_arr = np.array(y_test)
results = []
for k in range(1, k_neighbors+1):  # evaluate at k=1,2,3,4,5
    top_k_preds = neighbor_labels[:, :k]  # take first k neighbors
    
    correct    = [true in preds for true, preds in zip(y_test_arr, top_k_preds)]
    recall     = sum(correct) / len(correct)                      
    precision  = sum(correct) / (len(correct) * k)                
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    results.append({'k': k, 'precision': precision, 'recall': recall, 'f1': round(f1, 3)})
print(pd.DataFrame(results))

# --- Extra Visualizations & Summaries ---
viz_k = 6

# Use the top-viz_k neighbor as the predicted label (majority vote among viz_k neighbors)
top_k_preds = neighbor_labels[:, :viz_k]
y_pred = np.array([
    pd.Series(preds).mode()[0]  # majority vote; ties broken by first occurring label
    for preds in top_k_preds
])
labels = sorted(np.unique(np.concatenate([y_test_arr, y_pred])))

# Computing the F1 scores
f1_macro    = f1_score(y_test_arr, y_pred, average="macro",    zero_division=0)
f1_weighted = f1_score(y_test_arr, y_pred, average="weighted", zero_division=0)
print(f"\n--- Visualizations for k={viz_k} ---")
print(f"F1 (macro):    {f1_macro:.4f}")
print(f"F1 (weighted): {f1_weighted:.4f}")

# Compute per-class TP, FP, TN, FN then sum across all classes
mcm = multilabel_confusion_matrix(y_test_arr, y_pred, labels=labels)
total_TP = int(mcm[:, 1, 1].sum())  # TP : total correctly predicted as that class
total_FP = int(mcm[:, 0, 1].sum())  # FP : total incorrectly predicted as that class
total_TN = int(mcm[:, 0, 0].sum())  # TN : total correctly predicted as not that class
total_FN = int(mcm[:, 1, 0].sum())  # FN : total incorrectly predicted as not that class
summary = pd.DataFrame({
    "Metric": ["True Positives (TP)", "False Positives (FP)", "True Negatives (TN)", "False Negatives (FN)"],
    "Count": [total_TP, total_FP, total_TN, total_FN]
})
print("Confusion Matrix Summary:")
display(summary.style.hide(axis="index"))

    k  precision    recall     f1
0   1   0.777141  0.777141  0.777
1   2   0.419534  0.839067  0.559
2   3   0.288269  0.864806  0.432
3   4   0.220502  0.882008  0.353
4   5   0.178491  0.892457  0.297
5   6   0.150314  0.901886  0.258
6   7   0.129806  0.908639  0.227
7   8   0.114201  0.913609  0.203
8   9   0.101880  0.916922  0.183
9  10   0.091985  0.919852  0.167

--- Visualizations for k=6 ---
F1 (macro):    0.4523
F1 (weighted): 0.7282
Confusion Matrix Summary:


Metric,Count
True Positives (TP),5978
False Positives (FP),1870
True Negatives (TN),20756090
False Negatives (FN),1870


In [53]:
# Define once before the loop using the full label set
all_labels = sorted(np.unique(np.concatenate([y_test_arr, neighbor_labels.flatten()])))

for k in range(1, k_neighbors):
    top_k_preds = neighbor_labels[:, :k]

    correct   = [true in preds for true, preds in zip(y_test_arr, top_k_preds)]
    recall    = sum(correct) / len(correct)
    precision = sum(correct) / (len(correct) * k)
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results.append({'k': k, 'precision': precision, 'recall': recall, 'f1': round(f1, 3)})

    # Build y_pred consistent with the loop's definition of correct
    y_pred = np.array([
        true if true in preds else pd.Series(preds).mode()[0]
        for true, preds in zip(y_test_arr, top_k_preds)
    ])
    labels = sorted(np.unique(np.concatenate([y_test_arr, y_pred])))

    # F1 scores
    f1_macro    = f1_score(y_test_arr, y_pred, average="macro",    zero_division=0)
    f1_weighted = f1_score(y_test_arr, y_pred, average="weighted", zero_division=0)

    # TP/FP/TN/FN summary
    mcm      = multilabel_confusion_matrix(y_test_arr, y_pred, labels=all_labels)
    total_TP = int(mcm[:, 1, 1].sum())
    total_FP = int(mcm[:, 0, 1].sum())

    assert total_TP == sum(correct), f"k={k} TP mismatch: {total_TP} vs {sum(correct)}"

    summary = pd.DataFrame({
        "Metric": ["True Positives (TP)", "False Positives (FP)"],
        "Count":  [total_TP, total_FP]
    })

    print(f"\n{'='*60}")
    print(f"  k={k}  |  precision={precision:.4f}  recall={recall:.4f}  f1={f1:.4f}")
    print(f"  F1 macro={f1_macro:.4f}  F1 weighted={f1_weighted:.4f}")
    print(f"{'='*60}")
    print(f"Confusion Matrix (for {len(labels)} classes):")
    display(summary.style.hide(axis="index"))

print("\n--- Overall Eval Loop Results ---")
print(pd.DataFrame(results))


  k=1  |  precision=0.7771  recall=0.7771  f1=0.7771
  F1 macro=0.5005  F1 weighted=0.7719
Confusion Matrix (for 2925 classes):


Metric,Count
True Positives (TP),6099
False Positives (FP),1749



  k=2  |  precision=0.4195  recall=0.8391  f1=0.5594
  F1 macro=0.6119  F1 weighted=0.8266
Confusion Matrix (for 2707 classes):


Metric,Count
True Positives (TP),6585
False Positives (FP),1263



  k=3  |  precision=0.2883  recall=0.8648  f1=0.4324
  F1 macro=0.6672  F1 weighted=0.8503
Confusion Matrix (for 2621 classes):


...


  k=8  |  precision=0.1142  recall=0.9136  f1=0.2030
  F1 macro=0.7724  F1 weighted=0.9001
Confusion Matrix (for 2530 classes):


Metric,Count
True Positives (TP),7170
False Positives (FP),678



  k=9  |  precision=0.1019  recall=0.9169  f1=0.1834
  F1 macro=0.7815  F1 weighted=0.9034
Confusion Matrix (for 2517 classes):


Metric,Count
True Positives (TP),7196
False Positives (FP),652



--- Overall Eval Loop Results ---
     k  precision    recall     f1
0    1   0.777141  0.777141  0.777
1    2   0.419534  0.839067  0.559
2    3   0.288269  0.864806  0.432
3    4   0.220502  0.882008  0.353
4    5   0.178491  0.892457  0.297
5    6   0.150314  0.901886  0.258
6    7   0.129806  0.908639  0.227
7    8   0.114201  0.913609  0.203
8    9   0.101880  0.916922  0.183
9   10   0.091985  0.919852  0.167
10   1   0.777141  0.777141  0.777
11   2   0.419534  0.839067  0.559
12   3   0.288269  0.864806  0.432
13   4   0.220502  0.882008  0.353
14   5   0.178491  0.892457  0.297
15   1   0.777141  0.777141  0.777
16   2   0.419534  0.839067  0.559
17   3   0.288269  0.864806  0.432
18   4   0.220502  0.882008  0.353
19   5   0.178491  0.892457  0.297
20   6   0.150314  0.901886  0.258
21   7   0.129806  0.908639  0.227
22   8   0.114201  0.913609  0.203
23   9   0.101880  0.916922  0.183
24   1   0.777141  0.777141  0.777
25   2   0.419534  0.839067  0.559
26   3   0.288269  0

In [ ]:
import matplotlib.pyplot as plt

results_df = pd.DataFrame(results)

plt.figure(figsize=(7, 4))
for metric in ['precision', 'recall', 'f1']:
    plt.plot(results_df['k'], results_df[metric], marker='o', linewidth=2, markersize=6, label=metric)
plt.title('Metrics vs K')
plt.xlabel('k')
plt.ylabel('Score')
plt.xticks(results_df['k'])
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()